# MBEC V12A — post-processing audit of online-safe alarm streams

This notebook uses the saved **V11B stream-prediction parquet files**. It does
not fit or tune a classifier and does not alter the epoch predictions.

It produces the analyses required before manuscript revision:

1. exact regeneration of the causal 5-min, 21/30 alarm rule;
2. a 5-min minimum-warning-time sensitivity analysis;
3. TTW median, IQR, and the fraction of detections in the final 5 min;
4. mutually exclusive decomposition of all emitted alarms by retrospective state;
5. fold-stream alarm burden and an audit of repeated physical monitoring time;
6. patient-level macro metrics and patient-cluster bootstrap confidence intervals;
7. a patient-matched Poisson random-predictor benchmark with the same timeline,
   gaps, 30-min refractory period, and observed total alarm burden.

## Important terminology

The existing model was trained with a 30-min preictal interval ending at seizure
onset. Therefore, the V12A sensitivity interval is **−30 to −5 min**, which is a
25-min valid interval with a 5-min minimum warning time. It is not labelled
“SPH = 5 min, SOP = 30 min.” A formal SPH=5/SOP=30 analysis requires relabelling
the prediction interval as −35 to −5 min and belongs to V12B.

The random predictor is a rate-matched null benchmark, not a replacement for a
future fully prospective evaluation. Results are reported per patient and for
the cohort; empirical p-values are adjusted with Benjamini–Hochberg.


In [ ]:
# ============================================================================
# 0. Imports and locked V12A settings
# ============================================================================
import json
import math
import os
import platform
import sys
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

EPOCH_SEC = 10
SOP_MIN = 30
MIN_WARNING_MIN = 5
ALARM_WINDOW_MIN = 5
ALARM_WINDOW_EPOCHS = ALARM_WINDOW_MIN * 60 // EPOCH_SEC  # 30
ALARM_REQUIRED_POSITIVES = 21
REFRACTORY_MIN = 30
REFRACTORY_SEC = REFRACTORY_MIN * 60

RANDOM_SEED = 12042
N_MONTE_CARLO = int(os.getenv('V12A_MONTE_CARLO', '5000'))
N_CLUSTER_BOOTSTRAP = int(os.getenv('V12A_CLUSTER_BOOTSTRAP', '10000'))
STRICT = os.getenv('V12A_STRICT', '1') != '0'

OUTPUT_ROOT = Path('/kaggle/working/MBEC_V12A_POSTPROCESS')
if not Path('/kaggle/working').exists():
    OUTPUT_ROOT = Path.cwd() / 'MBEC_V12A_POSTPROCESS'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'figures').mkdir(exist_ok=True)

REQUIRED_COLUMNS = {
    'Subject', 'Model', 'FeatureSet', 'HeldOutSeizure', 'Epoch', 'EpochStart',
    'Phase', 'TargetPreictal', 'EligibleForFAR', 'Discontinuity', 'Prediction'
}

print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
print('Monte Carlo replicates:', N_MONTE_CARLO)
print('Cluster-bootstrap replicates:', N_CLUSTER_BOOTSTRAP)
print('Output:', OUTPUT_ROOT)


## 1. Locate and validate the V11B prediction streams

If automatic discovery selects the wrong dataset, set `V11B_ROOT` to the folder
that contains `stream_predictions/`. On Kaggle this can be done by editing the
variable below or setting the environment variable `V12A_INPUT_ROOT`.


In [ ]:
def discover_v11b_root():
    manual = os.getenv('V12A_INPUT_ROOT', '').strip()
    candidates = []
    if manual:
        candidates.append(Path(manual))
    candidates.extend([
        Path('/kaggle/working/MBEC_V11B_TARGETED_STREAM'),
        Path('/kaggle/input'),
        Path.cwd(),
    ])
    checked = []
    for candidate in candidates:
        if not candidate.exists():
            continue
        if candidate.name == 'stream_predictions':
            roots = [candidate.parent]
        elif (candidate / 'stream_predictions').exists():
            roots = [candidate]
        else:
            roots = sorted({p.parent for p in candidate.rglob('stream_predictions')})
        for root in roots:
            checked.append(root)
            files = list((root / 'stream_predictions').rglob('*__seizure_*.parquet'))
            if files:
                return root, sorted(files)
    raise FileNotFoundError(
        'No V11B stream parquets found. Attach the V11B output dataset and set '
        'V12A_INPUT_ROOT to the directory containing stream_predictions/. '
        f'Checked: {checked}'
    )


V11B_ROOT, PARQUET_FILES = discover_v11b_root()
print('V11B root:', V11B_ROOT)
print('Prediction parquet files:', len(PARQUET_FILES))

frames = []
file_qc = []
for path in PARQUET_FILES:
    frame = pd.read_parquet(path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    frame = frame.copy()
    frame['EpochStart'] = pd.to_datetime(frame['EpochStart'], utc=True)
    frame['SourceFile'] = str(path.relative_to(V11B_ROOT))
    frames.append(frame)
    file_qc.append({
        'SourceFile': frame['SourceFile'].iloc[0],
        'Rows': len(frame),
        'Subject': frame['Subject'].iloc[0],
        'Model': frame['Model'].iloc[0],
        'FeatureSet': frame['FeatureSet'].iloc[0],
        'HeldOutSeizure': int(frame['HeldOutSeizure'].iloc[0]),
    })

streams = pd.concat(frames, ignore_index=True)
streams['Phase'] = streams['Phase'].astype(str).str.lower()
for column in ['TargetPreictal', 'EligibleForFAR', 'Discontinuity', 'Prediction']:
    streams[column] = streams[column].astype(np.int8)

file_qc = pd.DataFrame(file_qc)
file_qc.to_csv(OUTPUT_ROOT / '01_input_file_qc.csv', index=False)

config_cols = ['Model', 'FeatureSet']
fold_cols = ['Subject', 'Model', 'FeatureSet', 'HeldOutSeizure']
configs = streams[config_cols].drop_duplicates().sort_values(config_cols)
fold_qc = streams.groupby(fold_cols, as_index=False).agg(
    Epochs=('Epoch', 'size'),
    TargetPreictalEpochs=('TargetPreictal', 'sum'),
    EligibleFAREpochs=('EligibleForFAR', 'sum'),
    FirstEpoch=('EpochStart', 'min'),
    LastEpoch=('EpochStart', 'max'),
)

duplicate_keys = streams.duplicated(fold_cols + ['EpochStart'], keep=False).sum()
if duplicate_keys:
    raise AssertionError(f'Duplicate timestamps inside fold streams: {duplicate_keys}')
if not set(streams['Prediction'].unique()).issubset({0, 1}):
    raise AssertionError('Prediction must be binary')
if STRICT and not (fold_qc['TargetPreictalEpochs'] == 180).all():
    bad = fold_qc.loc[fold_qc['TargetPreictalEpochs'] != 180]
    raise AssertionError(f'Expected 180 target-preictal epochs per fold:\n{bad}')

print('Configurations:', len(configs))
display(configs.reset_index(drop=True))
print('Subjects:', streams['Subject'].nunique())
print('Unique seizure folds per configuration:',
      fold_qc.groupby(config_cols).size().to_dict())
display(fold_qc.head())


## 2. Recreate the online-safe alarm generator

Only predictions, timestamps, discontinuities, and refractory state are used to
emit alarms. Retrospective labels are attached after emission for scoring.


In [ ]:
def infer_onset(stream):
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if target.empty:
        raise ValueError('No target-preictal epochs')
    return target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)


def generate_online_safe_alarms(stream):
    stream = stream.sort_values(['EpochStart', 'Epoch'], kind='stable').reset_index(drop=True)
    history = deque(maxlen=ALARM_WINDOW_EPOCHS)
    refractory_until = None
    previous_start = None
    last_boundary_time = None
    alarms = []

    for idx, row in stream.iterrows():
        start = row['EpochStart']
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row['Discontinuity'])
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        else:
            last_boundary_time = start
        previous_start = start

        if gap:
            history.clear()
            last_boundary_time = end
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        history.append(int(row['Prediction']))
        positive_count = int(sum(history))
        if len(history) == ALARM_WINDOW_EPOCHS and positive_count >= ALARM_REQUIRED_POSITIVES:
            since_boundary = np.nan
            if last_boundary_time is not None:
                since_boundary = (end - last_boundary_time).total_seconds() / 60.0
            alarms.append({
                'AlarmIndex': int(idx),
                'AlarmTime': end,
                'Phase': str(row['Phase']).lower(),
                'EligibleForFAR': int(row['EligibleForFAR']),
                'TargetPreictal': int(row['TargetPreictal']),
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / ALARM_WINDOW_EPOCHS,
                'MinutesSinceBoundary': since_boundary,
                'NearGapBoundary5min': int(np.isfinite(since_boundary) and since_boundary <= 5.01),
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def score_fold(stream):
    stream = stream.sort_values(['EpochStart', 'Epoch'], kind='stable').reset_index(drop=True)
    onset = infer_onset(stream)
    original_start = onset - pd.Timedelta(minutes=SOP_MIN)
    mwt_end = onset - pd.Timedelta(minutes=MIN_WARNING_MIN)
    alarms = generate_online_safe_alarms(stream)

    alarm_rows = []
    for order, alarm in enumerate(alarms, start=1):
        t = alarm['AlarmTime']
        rel_min = (t - onset).total_seconds() / 60.0
        is_original_true = original_start <= t < onset
        is_mwt5_true = original_start <= t < mwt_end
        is_last5 = mwt_end <= t < onset
        is_far = bool(alarm['EligibleForFAR'])

        if is_mwt5_true:
            category = 'preictal_valid_MWT5'
        elif is_last5:
            category = 'preictal_final_5min'
        elif is_far:
            category = 'eligible_interictal_FAR'
        elif alarm['Phase'] == 'ictal':
            category = 'ictal'
        elif alarm['Phase'] == 'postictal':
            category = 'postictal'
        elif alarm['Phase'] == 'gap':
            category = 'gap'
        elif alarm['Phase'] == 'excluded' and -35 <= rel_min < -30:
            category = 'excluded_5min_before_preictal'
        elif alarm['Phase'] == 'excluded':
            category = 'excluded_other'
        else:
            category = f'other_{alarm["Phase"]}'

        alarm_rows.append({
            **{c: stream[c].iloc[0] for c in fold_cols},
            'AlarmOrder': order,
            'SeizureOnset': onset,
            'RelativeToOnset_min': rel_min,
            'TTW_min': -rel_min if rel_min < 0 else np.nan,
            'OriginalTrueAlarm': int(is_original_true),
            'MWT5TrueAlarm': int(is_mwt5_true),
            'Final5minAlarm': int(is_last5),
            'FARScoredAlarm': int(is_far),
            'OriginalScoredAlarm': int(is_original_true or is_far),
            'OriginalUnscoredAlarm': int(not (is_original_true or is_far)),
            'AlarmCategory': category,
            **alarm,
        })

    first_original = next((a for a in alarm_rows if a['OriginalTrueAlarm']), None)
    first_mwt5 = next((a for a in alarm_rows if a['MWT5TrueAlarm']), None)
    fold_result = {
        **{c: stream[c].iloc[0] for c in fold_cols},
        'SeizureOnset': onset,
        'Epochs': len(stream),
        'UsableEpochs': int((stream['Discontinuity'] == 0).sum()),
        'FoldStreamHours': len(stream) * EPOCH_SEC / 3600.0,
        'UsableFoldStreamHours': (stream['Discontinuity'] == 0).sum() * EPOCH_SEC / 3600.0,
        'EligibleInterictalHours': stream['EligibleForFAR'].sum() * EPOCH_SEC / 3600.0,
        'TotalAlarms': len(alarm_rows),
        'FalseAlarms': int(sum(a['FARScoredAlarm'] for a in alarm_rows)),
        'OriginalUnscoredAlarms': int(sum(a['OriginalUnscoredAlarm'] for a in alarm_rows)),
        'Predicted_SPH0': int(first_original is not None),
        'Predicted_MWT5': int(first_mwt5 is not None),
        'DetectionOnlyFinal5min': int(first_original is not None and first_mwt5 is None),
        'TTW_SPH0_min': first_original['TTW_min'] if first_original else np.nan,
        'TTW_MWT5_min': first_mwt5['TTW_min'] if first_mwt5 else np.nan,
    }
    return fold_result, alarm_rows


def unit_test_alarm_logic():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset - pd.Timedelta(minutes=40), periods=240, freq='10s')
    toy = pd.DataFrame({
        'Subject': 'toy', 'Model': 'toy', 'FeatureSet': 'toy',
        'HeldOutSeizure': 1, 'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['excluded'] * 60 + ['preictal'] * 180,
        'TargetPreictal': [0] * 60 + [1] * 180,
        'EligibleForFAR': 0, 'Discontinuity': 0,
        'Prediction': [0] * 60 + [1] * 180,
    })
    result, alarms = score_fold(toy)
    assert result['Predicted_SPH0'] == 1
    assert result['Predicted_MWT5'] == 1
    assert len(alarms) == 1
    broken = toy.copy()
    broken.loc[70, 'Discontinuity'] = 1
    broken['Prediction'] = 0
    broken.loc[60:89, 'Prediction'] = 1
    result, _ = score_fold(broken)
    assert result['Predicted_SPH0'] == 0
    print('Alarm-generator unit tests: PASS')


unit_test_alarm_logic()


## 3. Regenerate every alarm and audit against V11B


In [ ]:
fold_results = []
alarm_events = []
for _, group in streams.groupby(fold_cols, sort=True):
    fold_result, emitted = score_fold(group)
    fold_results.append(fold_result)
    alarm_events.extend(emitted)

fold_results = pd.DataFrame(fold_results)
alarm_events = pd.DataFrame(alarm_events)
if alarm_events.empty:
    alarm_events = pd.DataFrame(columns=[
        *fold_cols, 'AlarmTime', 'AlarmCategory', 'OriginalTrueAlarm',
        'MWT5TrueAlarm', 'FARScoredAlarm', 'OriginalUnscoredAlarm'
    ])

fold_results.to_csv(OUTPUT_ROOT / '02_fold_metrics_regenerated.csv', index=False)
alarm_events.to_csv(OUTPUT_ROOT / '03_all_emitted_alarms.csv', index=False)

# Optional exact cross-check against the V11B primary online-safe CSV.
v11b_alarm_csv = V11B_ROOT / 'targeted_alarm_sensitivity_folds.csv'
crosscheck_rows = []
if v11b_alarm_csv.exists():
    old = pd.read_csv(v11b_alarm_csv)
    old = old.loc[
        old['IsPrimary'].astype(bool) &
        old['AlarmGenerationMode'].eq('online_safe_gap_only')
    ].copy()
    merged = fold_results.merge(old, on=fold_cols, suffixes=('_V12A', '_V11B'), how='outer', indicator=True)
    for field_v12, field_v11 in [
        ('TotalAlarms_V12A', 'TotalAlarms_V11B'),
        ('FalseAlarms_V12A', 'FalseAlarms_V11B'),
        ('Predicted_SPH0', 'SeizurePredicted'),
    ]:
        if field_v12 in merged and field_v11 in merged:
            merged[f'Match_{field_v12}'] = merged[field_v12].eq(merged[field_v11])
    merged.to_csv(OUTPUT_ROOT / '04_v11b_regeneration_crosscheck.csv', index=False)
    match_cols = [c for c in merged if c.startswith('Match_')]
    if match_cols and not merged[match_cols].all().all():
        raise AssertionError('V12A regeneration does not exactly match V11B; inspect crosscheck CSV')
    print('V11B primary online-safe regeneration: EXACT MATCH')
else:
    print('V11B alarm CSV not attached; parquet-only regeneration completed.')

display(fold_results.groupby(config_cols).agg(
    Seizures=('HeldOutSeizure', 'size'),
    Predicted_SPH0=('Predicted_SPH0', 'sum'),
    Predicted_MWT5=('Predicted_MWT5', 'sum'),
    FalseAlarms=('FalseAlarms', 'sum'),
    TotalAlarms=('TotalAlarms', 'sum'),
    OriginalUnscored=('OriginalUnscoredAlarms', 'sum'),
))


## 4. Warning-time and alarm-state analyses

Alarm categories are mutually exclusive. “Original unscored” means that an
alarm was neither inside the original −30 to 0 min prediction interval nor
inside an epoch eligible for FAR.


In [ ]:
def q25(series): return series.quantile(0.25)
def q75(series): return series.quantile(0.75)

warning_summary = fold_results.groupby(config_cols, as_index=False).agg(
    Subjects=('Subject', 'nunique'),
    Seizures=('HeldOutSeizure', 'size'),
    Predicted_SPH0=('Predicted_SPH0', 'sum'),
    Predicted_MWT5=('Predicted_MWT5', 'sum'),
    DetectionOnlyFinal5min=('DetectionOnlyFinal5min', 'sum'),
    MeanTTW_SPH0_min=('TTW_SPH0_min', 'mean'),
    MedianTTW_SPH0_min=('TTW_SPH0_min', 'median'),
    Q1TTW_SPH0_min=('TTW_SPH0_min', q25),
    Q3TTW_SPH0_min=('TTW_SPH0_min', q75),
    MinTTW_SPH0_min=('TTW_SPH0_min', 'min'),
    MaxTTW_SPH0_min=('TTW_SPH0_min', 'max'),
)
warning_summary['Sensitivity_SPH0'] = warning_summary['Predicted_SPH0'] / warning_summary['Seizures']
warning_summary['Sensitivity_MWT5'] = warning_summary['Predicted_MWT5'] / warning_summary['Seizures']
warning_summary['FractionDetectedOnlyFinal5min_among_SPH0'] = (
    warning_summary['DetectionOnlyFinal5min'] / warning_summary['Predicted_SPH0'].replace(0, np.nan)
)
warning_summary.to_csv(OUTPUT_ROOT / '05_warning_time_sensitivity_summary.csv', index=False)

state_breakdown = (
    alarm_events.groupby(config_cols + ['AlarmCategory'], as_index=False)
    .size().rename(columns={'size': 'Alarms'})
)
state_totals = state_breakdown.groupby(config_cols)['Alarms'].transform('sum')
state_breakdown['PercentOfAllAlarms'] = 100 * state_breakdown['Alarms'] / state_totals
state_breakdown.to_csv(OUTPUT_ROOT / '06_alarm_state_breakdown.csv', index=False)

phase_breakdown = (
    alarm_events.groupby(config_cols + ['Phase'], as_index=False)
    .agg(Alarms=('AlarmTime', 'size'), NearGapBoundary5min=('NearGapBoundary5min', 'sum'))
)
phase_breakdown.to_csv(OUTPUT_ROOT / '07_alarm_phase_and_gap_breakdown.csv', index=False)

display(warning_summary)
display(state_breakdown)


## 5. Monitoring-time overlap and total alarm burden

`FoldStreamAlarmRate` is the valid alarm-burden measure for the current
held-out-seizure evaluation. `OverlapFactor` quantifies how often physical EEG
time is represented across fold-specific streams. The exploratory deduplicated
rate must not be described as a prospective patient-day rate because alarms can
come from different fold-specific fitted models.


In [ ]:
time_rows = []
for keys, group in streams.groupby(config_cols, sort=True):
    model, feature = keys
    fold_epoch_count = len(group)
    unique_epoch_count = group[['Subject', 'EpochStart']].drop_duplicates().shape[0]
    duplicated_epoch_count = fold_epoch_count - unique_epoch_count
    events = alarm_events.loc[
        alarm_events['Model'].eq(model) & alarm_events['FeatureSet'].eq(feature)
    ]
    unique_events = events[['Subject', 'AlarmTime']].drop_duplicates().shape[0]
    fold_hours = fold_epoch_count * EPOCH_SEC / 3600.0
    unique_hours = unique_epoch_count * EPOCH_SEC / 3600.0
    time_rows.append({
        'Model': model, 'FeatureSet': feature,
        'FoldStreamEpochs': fold_epoch_count,
        'UniquePhysicalEpochs': unique_epoch_count,
        'RepeatedFoldStreamEpochs': duplicated_epoch_count,
        'OverlapFactor': fold_epoch_count / unique_epoch_count,
        'FoldStreamHours': fold_hours,
        'UniquePhysicalHours': unique_hours,
        'TotalAlarmEventsAcrossFolds': len(events),
        'ExactTimestampDeduplicatedAlarmEvents': unique_events,
        'FoldStreamAlarmRate_per_h': len(events) / fold_hours if fold_hours else np.nan,
        'ExploratoryDeduplicatedAlarmRate_per_h': unique_events / unique_hours if unique_hours else np.nan,
    })

monitoring_audit = pd.DataFrame(time_rows)
monitoring_audit.to_csv(OUTPUT_ROOT / '08_monitoring_time_overlap_and_alarm_burden.csv', index=False)

subject_burden = fold_results.groupby(['Subject'] + config_cols, as_index=False).agg(
    Seizures=('HeldOutSeizure', 'size'),
    FoldStreamHours=('FoldStreamHours', 'sum'),
    EligibleInterictalHours=('EligibleInterictalHours', 'sum'),
    TotalAlarms=('TotalAlarms', 'sum'),
    FalseAlarms=('FalseAlarms', 'sum'),
    Predicted_SPH0=('Predicted_SPH0', 'sum'),
    Predicted_MWT5=('Predicted_MWT5', 'sum'),
)
subject_burden['Sensitivity_SPH0'] = subject_burden['Predicted_SPH0'] / subject_burden['Seizures']
subject_burden['Sensitivity_MWT5'] = subject_burden['Predicted_MWT5'] / subject_burden['Seizures']
subject_burden['FAR_per_h'] = subject_burden['FalseAlarms'] / subject_burden['EligibleInterictalHours'].replace(0, np.nan)
subject_burden['TotalAlarmRate_per_h'] = subject_burden['TotalAlarms'] / subject_burden['FoldStreamHours'].replace(0, np.nan)
subject_burden.to_csv(OUTPUT_ROOT / '09_patient_level_metrics.csv', index=False)

display(monitoring_audit)
display(subject_burden)


## 6. Patient-cluster bootstrap

Patients, rather than seizures, are resampled with replacement. Every sampled
patient contributes all of their seizures and eligible interictal hours.


In [ ]:
def cluster_bootstrap(group, n_boot=N_CLUSTER_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    subjects = np.array(sorted(group['Subject'].unique()))
    by_subject = {s: group.loc[group['Subject'].eq(s)] for s in subjects}
    records = []
    for _ in range(n_boot):
        sampled = rng.choice(subjects, size=len(subjects), replace=True)
        chunks = [by_subject[s] for s in sampled]
        seizures = sum(len(x) for x in chunks)
        pred0 = sum(int(x['Predicted_SPH0'].sum()) for x in chunks)
        pred5 = sum(int(x['Predicted_MWT5'].sum()) for x in chunks)
        false = sum(int(x['FalseAlarms'].sum()) for x in chunks)
        hours = sum(float(x['EligibleInterictalHours'].sum()) for x in chunks)
        per_patient_ss0 = [x['Predicted_SPH0'].mean() for x in chunks]
        per_patient_ss5 = [x['Predicted_MWT5'].mean() for x in chunks]
        per_patient_far = [
            x['FalseAlarms'].sum() / x['EligibleInterictalHours'].sum()
            if x['EligibleInterictalHours'].sum() > 0 else np.nan
            for x in chunks
        ]
        records.append((
            pred0 / seizures, pred5 / seizures, false / hours if hours else np.nan,
            np.mean(per_patient_ss0), np.mean(per_patient_ss5), np.nanmean(per_patient_far)
        ))
    columns = [
        'PooledSensitivity_SPH0', 'PooledSensitivity_MWT5', 'PooledFAR_per_h',
        'MacroSensitivity_SPH0', 'MacroSensitivity_MWT5', 'MacroFAR_per_h'
    ]
    return pd.DataFrame(records, columns=columns)


cluster_rows = []
for config_index, (keys, group) in enumerate(fold_results.groupby(config_cols, sort=True)):
    boot = cluster_bootstrap(group, seed=RANDOM_SEED + config_index)
    subject_group = subject_burden.loc[
        subject_burden['Model'].eq(keys[0]) & subject_burden['FeatureSet'].eq(keys[1])
    ]
    observed = {
        'PooledSensitivity_SPH0': group['Predicted_SPH0'].mean(),
        'PooledSensitivity_MWT5': group['Predicted_MWT5'].mean(),
        'PooledFAR_per_h': group['FalseAlarms'].sum() / group['EligibleInterictalHours'].sum(),
        'MacroSensitivity_SPH0': subject_group['Sensitivity_SPH0'].mean(),
        'MacroSensitivity_MWT5': subject_group['Sensitivity_MWT5'].mean(),
        'MacroFAR_per_h': subject_group['FAR_per_h'].mean(),
    }
    row = {'Model': keys[0], 'FeatureSet': keys[1], 'Subjects': group['Subject'].nunique(),
           'Seizures': len(group)}
    for metric, value in observed.items():
        row[metric] = value
        row[f'{metric}_CI95_low'] = boot[metric].quantile(0.025)
        row[f'{metric}_CI95_high'] = boot[metric].quantile(0.975)
    cluster_rows.append(row)

cluster_summary = pd.DataFrame(cluster_rows)
cluster_summary.to_csv(OUTPUT_ROOT / '10_patient_cluster_bootstrap_summary.csv', index=False)
display(cluster_summary)


## 7. Rate-matched Poisson random-predictor benchmark

For each patient and configuration, the observed total emitted-alarm rate is
converted to the latent Poisson rate of a non-paralyzable process with a 30-min
dead time:

\[
\lambda = \frac{r}{1-rd},
\]

where (r) is the observed emitted-alarm rate and (d=0.5) h. Random alarms
are generated on the actual contiguous stream segments, retaining gaps and the
same refractory duration. The null is deliberately matched on **total alarm
burden**, rather than FAR alone.


In [ ]:
def contiguous_segments(stream):
    stream = stream.sort_values(['EpochStart', 'Epoch'], kind='stable').reset_index(drop=True)
    starts = stream['EpochStart'].to_numpy(dtype='datetime64[ns]')
    discontinuity = stream['Discontinuity'].astype(bool).to_numpy()
    breaks = np.ones(len(stream), dtype=bool)
    if len(stream) > 1:
        delta_sec = np.diff(starts).astype('timedelta64[ns]').astype(np.int64) / 1e9
        breaks[1:] = (delta_sec > EPOCH_SEC + 0.5) | (delta_sec <= 0)
    breaks |= discontinuity
    segments = []
    current = []
    for idx in range(len(stream)):
        if discontinuity[idx]:
            if current:
                segments.append(np.asarray(current, dtype=int)); current = []
            continue
        if breaks[idx] and current:
            segments.append(np.asarray(current, dtype=int)); current = []
        current.append(idx)
    if current:
        segments.append(np.asarray(current, dtype=int))
    return stream, segments


def prepare_random_fold(stream):
    stream, segments = contiguous_segments(stream)
    onset = infer_onset(stream)
    original_start = onset - pd.Timedelta(minutes=SOP_MIN)
    mwt_end = onset - pd.Timedelta(minutes=MIN_WARNING_MIN)
    ends_ns = (stream['EpochStart'] + pd.Timedelta(seconds=EPOCH_SEC)).astype('int64').to_numpy()
    return {
        'stream': stream,
        'segments': segments,
        'ends_ns': ends_ns,
        'onset_ns': onset.value,
        'original_start_ns': original_start.value,
        'mwt_end_ns': mwt_end.value,
        'eligible': stream['EligibleForFAR'].astype(bool).to_numpy(),
    }


def simulate_random_fold(prepared, latent_rate_h, rng):
    if latent_rate_h <= 0:
        return 0, 0, 0, 0
    rate_per_ns = latent_rate_h / (3600.0 * 1e9)
    refractory_ns = int(REFRACTORY_SEC * 1e9)
    refractory_until = -np.inf
    predicted0 = 0
    predicted5 = 0
    false_alarms = 0
    total_alarms = 0

    for indices in prepared['segments']:
        seg_ends = prepared['ends_ns'][indices]
        seg_start = int(seg_ends[0] - EPOCH_SEC * 1e9)
        seg_end = int(seg_ends[-1])
        cursor = max(float(seg_start), float(refractory_until))
        while True:
            cursor += rng.exponential(1.0 / rate_per_ns)
            if cursor >= seg_end:
                break
            local = int(np.searchsorted(seg_ends, cursor, side='left'))
            if local >= len(indices):
                break
            idx = int(indices[local])
            alarm_ns = int(prepared['ends_ns'][idx])
            if alarm_ns < refractory_until:
                cursor = float(refractory_until)
                continue
            total_alarms += 1
            false_alarms += int(prepared['eligible'][idx])
            predicted0 |= int(prepared['original_start_ns'] <= alarm_ns < prepared['onset_ns'])
            predicted5 |= int(prepared['original_start_ns'] <= alarm_ns < prepared['mwt_end_ns'])
            refractory_until = alarm_ns + refractory_ns
            cursor = float(refractory_until)
    return int(predicted0), int(predicted5), int(false_alarms), int(total_alarms)


prepared_folds = {}
for keys, group in streams.groupby(fold_cols, sort=True):
    prepared_folds[tuple(keys)] = prepare_random_fold(group)


def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    n = len(pvalues)
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    adjusted = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n, dtype=float)
    out[order] = adjusted
    return out


rng = np.random.default_rng(RANDOM_SEED)
patient_chance_rows = []
cohort_chance_rows = []

for config_index, (config, cfg_folds) in enumerate(fold_results.groupby(config_cols, sort=True)):
    model, feature = config
    patient_simulations = {}
    patient_observed = {}
    patient_rates = {}
    for subject, patient in cfg_folds.groupby('Subject', sort=True):
        observed_rate = patient['TotalAlarms'].sum() / patient['UsableFoldStreamHours'].sum()
        dead_time_h = REFRACTORY_MIN / 60.0
        if observed_rate * dead_time_h >= 1:
            raise ValueError(f'Observed alarm rate incompatible with refractory model: {subject}, {config}')
        latent_rate = observed_rate / max(1e-12, 1 - observed_rate * dead_time_h)
        patient_rates[subject] = (observed_rate, latent_rate)
        patient_observed[subject] = {
            'n': len(patient),
            'ss0': patient['Predicted_SPH0'].mean(),
            'ss5': patient['Predicted_MWT5'].mean(),
        }
        patient_simulations[subject] = np.zeros((N_MONTE_CARLO, 4), dtype=float)

    for rep in range(N_MONTE_CARLO):
        for subject, patient in cfg_folds.groupby('Subject', sort=True):
            latent_rate = patient_rates[subject][1]
            totals = np.zeros(4, dtype=float)
            for row in patient.itertuples(index=False):
                key = (row.Subject, row.Model, row.FeatureSet, row.HeldOutSeizure)
                totals += simulate_random_fold(prepared_folds[key], latent_rate, rng)
            totals[0] /= len(patient)
            totals[1] /= len(patient)
            patient_simulations[subject][rep] = totals

    cohort_null0 = np.zeros(N_MONTE_CARLO)
    cohort_null5 = np.zeros(N_MONTE_CARLO)
    total_seizures = len(cfg_folds)
    for subject, sim in patient_simulations.items():
        n = patient_observed[subject]['n']
        cohort_null0 += sim[:, 0] * n / total_seizures
        cohort_null5 += sim[:, 1] * n / total_seizures
        obs = patient_observed[subject]
        p0 = (1 + np.sum(sim[:, 0] >= obs['ss0'] - 1e-12)) / (N_MONTE_CARLO + 1)
        p5 = (1 + np.sum(sim[:, 1] >= obs['ss5'] - 1e-12)) / (N_MONTE_CARLO + 1)
        patient_chance_rows.append({
            'Model': model, 'FeatureSet': feature, 'Subject': subject,
            'Seizures': n,
            'ObservedAlarmRate_per_h': patient_rates[subject][0],
            'LatentPoissonRate_per_h': patient_rates[subject][1],
            'ObservedSensitivity_SPH0': obs['ss0'],
            'ChanceMeanSensitivity_SPH0': sim[:, 0].mean(),
            'ChanceCI95Low_SPH0': np.quantile(sim[:, 0], 0.025),
            'ChanceCI95High_SPH0': np.quantile(sim[:, 0], 0.975),
            'EmpiricalP_SPH0': p0,
            'ObservedSensitivity_MWT5': obs['ss5'],
            'ChanceMeanSensitivity_MWT5': sim[:, 1].mean(),
            'ChanceCI95Low_MWT5': np.quantile(sim[:, 1], 0.025),
            'ChanceCI95High_MWT5': np.quantile(sim[:, 1], 0.975),
            'EmpiricalP_MWT5': p5,
        })

    observed0 = cfg_folds['Predicted_SPH0'].mean()
    observed5 = cfg_folds['Predicted_MWT5'].mean()
    cohort_chance_rows.append({
        'Model': model, 'FeatureSet': feature,
        'Subjects': cfg_folds['Subject'].nunique(), 'Seizures': total_seizures,
        'ObservedSensitivity_SPH0': observed0,
        'ChanceMeanSensitivity_SPH0': cohort_null0.mean(),
        'ChanceCI95Low_SPH0': np.quantile(cohort_null0, 0.025),
        'ChanceCI95High_SPH0': np.quantile(cohort_null0, 0.975),
        'EmpiricalP_SPH0': (1 + np.sum(cohort_null0 >= observed0 - 1e-12)) / (N_MONTE_CARLO + 1),
        'ObservedSensitivity_MWT5': observed5,
        'ChanceMeanSensitivity_MWT5': cohort_null5.mean(),
        'ChanceCI95Low_MWT5': np.quantile(cohort_null5, 0.025),
        'ChanceCI95High_MWT5': np.quantile(cohort_null5, 0.975),
        'EmpiricalP_MWT5': (1 + np.sum(cohort_null5 >= observed5 - 1e-12)) / (N_MONTE_CARLO + 1),
    })

patient_chance = pd.DataFrame(patient_chance_rows)
for config, idx in patient_chance.groupby(config_cols).groups.items():
    patient_chance.loc[idx, 'BH_q_SPH0'] = benjamini_hochberg(patient_chance.loc[idx, 'EmpiricalP_SPH0'])
    patient_chance.loc[idx, 'BH_q_MWT5'] = benjamini_hochberg(patient_chance.loc[idx, 'EmpiricalP_MWT5'])
patient_chance['ExceedsChance_SPH0_q05'] = (patient_chance['BH_q_SPH0'] < 0.05).astype(int)
patient_chance['ExceedsChance_MWT5_q05'] = (patient_chance['BH_q_MWT5'] < 0.05).astype(int)

cohort_chance = pd.DataFrame(cohort_chance_rows)
counts = patient_chance.groupby(config_cols, as_index=False).agg(
    PatientsExceedChance_SPH0_q05=('ExceedsChance_SPH0_q05', 'sum'),
    PatientsExceedChance_MWT5_q05=('ExceedsChance_MWT5_q05', 'sum'),
)
cohort_chance = cohort_chance.merge(counts, on=config_cols, how='left')

patient_chance.to_csv(OUTPUT_ROOT / '11_random_predictor_patient_results.csv', index=False)
cohort_chance.to_csv(OUTPUT_ROOT / '12_random_predictor_cohort_results.csv', index=False)
display(cohort_chance)
display(patient_chance)


## 8. Publication-quality audit figures and run manifest


In [ ]:
# Short labels prevent collisions in both single- and double-column layouts.
sns.set_theme(style='whitegrid', context='paper', font_scale=1.08)
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'axes.titleweight': 'bold',
    'axes.edgecolor': '#333333', 'grid.color': '#D9DEE3',
    'grid.linewidth': 0.7, 'legend.frameon': False,
})

CONFIG_LABEL = {
    ('RandomForest', 'F1_Handcrafted'): 'RF–F1',
    ('RandomForest', 'F7_All'): 'RF–F7',
    ('XGBoost', 'F1_Handcrafted'): 'XGB–F1',
    ('XGBoost', 'F6_Handcrafted_wPLI'): 'XGB–F6',
}
CONFIG_ORDER = ['RF–F1', 'RF–F7', 'XGB–F1', 'XGB–F6']
CONFIG_COLOR = {
    'RF–F1': '#2B6EA6', 'RF–F7': '#5B8FC0',
    'XGB–F1': '#C66B2B', 'XGB–F6': '#E09A58',
}
CATEGORY_ORDER = [
    'eligible_interictal_FAR', 'preictal_valid_MWT5', 'preictal_final_5min',
    'excluded_5min_before_preictal', 'excluded_other', 'postictal'
]
CATEGORY_LABEL = {
    'eligible_interictal_FAR': 'Eligible interictal (FAR)',
    'preictal_valid_MWT5': 'Valid preictal (≥5 min)',
    'preictal_final_5min': 'Final 5 min',
    'excluded_5min_before_preictal': 'Excluded: −35 to −30 min',
    'excluded_other': 'Other excluded time',
    'postictal': 'Postictal',
}
CATEGORY_COLOR = {
    'eligible_interictal_FAR': '#2878B5',
    'preictal_valid_MWT5': '#58B7C8',
    'preictal_final_5min': '#D4B52C',
    'excluded_5min_before_preictal': '#55A868',
    'excluded_other': '#8C6BB1',
    'postictal': '#D66AB3',
}

chance_plot = cohort_chance.copy()
chance_plot['Configuration'] = [
    CONFIG_LABEL[(m, f)] for m, f in zip(chance_plot['Model'], chance_plot['FeatureSet'])
]
chance_plot['Configuration'] = pd.Categorical(
    chance_plot['Configuration'], categories=CONFIG_ORDER, ordered=True)
chance_plot = chance_plot.sort_values('Configuration')

detected = fold_results.loc[fold_results['Predicted_SPH0'].astype(bool)].copy()
detected['Configuration'] = [
    CONFIG_LABEL[(m, f)] for m, f in zip(detected['Model'], detected['FeatureSet'])
]
detected['Configuration'] = pd.Categorical(
    detected['Configuration'], categories=CONFIG_ORDER, ordered=True)

state_plot = state_breakdown.copy()
state_plot['Configuration'] = [
    CONFIG_LABEL[(m, f)] for m, f in zip(state_plot['Model'], state_plot['FeatureSet'])
]
state_plot = state_plot.pivot_table(
    index='Configuration', columns='AlarmCategory', values='Alarms', fill_value=0,
    observed=False
).reindex(CONFIG_ORDER).reindex(columns=CATEGORY_ORDER, fill_value=0)


def draw_chance_panel(ax, panel_label=None):
    x = np.arange(len(chance_plot))
    width = 0.34
    observed = chance_plot['ObservedSensitivity_MWT5'].to_numpy()
    chance = chance_plot['ChanceMeanSensitivity_MWT5'].to_numpy()
    low = chance - chance_plot['ChanceCI95Low_MWT5'].to_numpy()
    high = chance_plot['ChanceCI95High_MWT5'].to_numpy() - chance
    ax.bar(x-width/2, observed, width, color='#2166AC', label='Observed', zorder=3)
    ax.bar(x+width/2, chance, width, color='#A6A6A6', label='Random predictor', zorder=3)
    ax.errorbar(x+width/2, chance, yerr=np.vstack([low, high]), fmt='none',
                ecolor='#4D4D4D', elinewidth=1.0, capsize=3, zorder=4)
    for i, row in enumerate(chance_plot.itertuples(index=False)):
        p = row.EmpiricalP_MWT5
        text = f'p={p:.3f}' if p >= 0.001 else 'p<0.001'
        ax.text(i, max(observed[i], chance[i]+high[i])+0.018, text,
                ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x, chance_plot['Configuration'])
    ax.set_ylim(0, max(0.56, float((observed+0.10).max())))
    ax.set_ylabel('Seizure sensitivity')
    ax.set_xlabel('Configuration')
    ax.set_title(f'{panel_label}Observed versus rate-matched chance' if panel_label else
                 'Observed versus rate-matched chance', loc='left')
    ax.legend(loc='upper right', ncol=1)
    ax.grid(axis='x', visible=False)


def draw_ttw_panel(ax, panel_label=None):
    sns.boxplot(data=detected, x='Configuration', y='TTW_SPH0_min',
                order=CONFIG_ORDER, palette=CONFIG_COLOR, hue='Configuration',
                legend=False, showfliers=False, width=0.58, linewidth=1.0, ax=ax)
    sns.stripplot(data=detected, x='Configuration', y='TTW_SPH0_min',
                  order=CONFIG_ORDER, color='#1F2933', alpha=0.55,
                  jitter=0.16, size=3.2, ax=ax)
    ax.axhline(MIN_WARNING_MIN, color='#C62828', linestyle='--', linewidth=1.1)
    ax.text(0.98, MIN_WARNING_MIN + 0.35, '5-min minimum warning',
            transform=ax.get_yaxis_transform(), ha='right', va='bottom',
            fontsize=8.5, color='#A61B1B',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.88, pad=1.5))
    ax.set_ylabel('Time to warning (min)')
    ax.set_xlabel('Configuration')
    ax.set_title(f'{panel_label}Time-to-warning distribution' if panel_label else
                 'Time-to-warning distribution', loc='left')
    ax.grid(axis='x', visible=False)


def draw_alarm_panel(ax, panel_label=None):
    y = np.arange(len(CONFIG_ORDER))
    left = np.zeros(len(CONFIG_ORDER))
    for category in CATEGORY_ORDER:
        values = state_plot[category].to_numpy(dtype=float)
        ax.barh(y, values, left=left, height=0.62,
                color=CATEGORY_COLOR[category], label=CATEGORY_LABEL[category])
        left += values
    for yi, total in enumerate(left):
        ax.text(total+2.0, yi, f'n={int(total)}', va='center', ha='left', fontsize=8.5)
    ax.set_yticks(y, CONFIG_ORDER)
    ax.invert_yaxis()
    ax.set_xlim(0, max(left)*1.12)
    ax.set_xlabel('Number of emitted alarms')
    ax.set_ylabel('Configuration')
    ax.set_title(f'{panel_label}Retrospective decomposition of emitted alarms' if panel_label else
                 'Retrospective decomposition of emitted alarms', loc='left')
    ax.grid(axis='y', visible=False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3,
              columnspacing=1.2, handlelength=1.5)


# Main three-panel figure: two compact panels above and one wide panel below.
fig = plt.figure(figsize=(10.6, 7.5), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.08], hspace=0.16, wspace=0.20)
ax_a = fig.add_subplot(grid[0, 0])
ax_b = fig.add_subplot(grid[0, 1])
ax_c = fig.add_subplot(grid[1, :])
draw_chance_panel(ax_a, '(a) ')
draw_ttw_panel(ax_b, '(b) ')
draw_alarm_panel(ax_c, '(c) ')
fig.savefig(OUTPUT_ROOT / 'figures' / 'Fig_V12A_audit_revised.png',
            dpi=400, bbox_inches='tight', facecolor='white')
fig.savefig(OUTPUT_ROOT / 'figures' / 'Fig_V12A_audit_revised.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()

# Standalone panels provide flexible placement in the main article or ESM.
for filename, drawer, size in [
    ('Fig_V12A_chance', draw_chance_panel, (6.6, 4.2)),
    ('Fig_V12A_TTW', draw_ttw_panel, (6.6, 4.2)),
    ('Fig_V12A_alarm_breakdown', draw_alarm_panel, (8.2, 4.6)),
]:
    panel_fig, panel_ax = plt.subplots(figsize=size, constrained_layout=True)
    drawer(panel_ax)
    panel_fig.savefig(OUTPUT_ROOT / 'figures' / f'{filename}.png',
                      dpi=400, bbox_inches='tight', facecolor='white')
    panel_fig.savefig(OUTPUT_ROOT / 'figures' / f'{filename}.pdf',
                      bbox_inches='tight', facecolor='white')
    plt.show()

manifest = {
    'analysis': 'MBEC V12A post-processing audit',
    'input_root': str(V11B_ROOT),
    'parquet_files': len(PARQUET_FILES),
    'rows': int(len(streams)),
    'subjects': int(streams['Subject'].nunique()),
    'configurations': int(len(configs)),
    'seizure_folds_per_configuration': {
        f'{k[0]}::{k[1]}': int(v)
        for k, v in fold_results.groupby(config_cols).size().items()
    },
    'epoch_sec': EPOCH_SEC,
    'original_preictal_min': SOP_MIN,
    'minimum_warning_min': MIN_WARNING_MIN,
    'alarm_window_epochs': ALARM_WINDOW_EPOCHS,
    'required_positive_epochs': ALARM_REQUIRED_POSITIVES,
    'refractory_min': REFRACTORY_MIN,
    'monte_carlo_replicates': N_MONTE_CARLO,
    'cluster_bootstrap_replicates': N_CLUSTER_BOOTSTRAP,
    'random_seed': RANDOM_SEED,
    'outputs': sorted(p.name for p in OUTPUT_ROOT.glob('*.csv')),
}
(OUTPUT_ROOT / 'V12A_run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('\nV12A COMPLETE')
print('Results:', OUTPUT_ROOT)
print('Important manuscript tables:')
for name in [
    '05_warning_time_sensitivity_summary.csv',
    '06_alarm_state_breakdown.csv',
    '08_monitoring_time_overlap_and_alarm_burden.csv',
    '10_patient_cluster_bootstrap_summary.csv',
    '11_random_predictor_patient_results.csv',
    '12_random_predictor_cohort_results.csv',
]:
    print(' -', name)
